# TBN-MAND ID/OOD 분포 시각화

**설정**: 셀 1에서 `SEED` / `INCREMENT` / `TASK_IDS` 설정 → 가중치 경로가 자동으로 맞춰짐. `TASK_IDS=None`이면 inc8→0–2, inc4→0–6, inc2→0–15 전부 저장.
- 가중치: `beta0.005_lambda0.4`
- OOD **3종만** (MAND에 ConfNormalized/4번째 없음): Main Only / Uniform Sum / Uniform Average → **1×3** 그림
- **중요**: 셀 1에서 `OOD_METHODS`를 바꾼 뒤에는 **반드시 셀 2까지 다시 실행**해야 합니다. 셀 2를 생략하면 `model`에 예전 `ood_methods`(4종)가 남아 Adaptive Reliability까지 다시 평가됩니다. 헷갈리면 **Restart Kernel 후 위에서부터 순서대로** 실행하세요.


In [9]:
# ===== 셀 1: 설정 =====
import os, sys, ast, copy, json
import numpy as np
import torch
import warnings
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore', message='.*weights_only.*', category=FutureWarning)

PROJECT_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebook' else os.getcwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# ── 변경 가능한 파라미터 ──────────────────────────────────────
SEED      = 1997
INCREMENT = 2
# None → INCREMENT에 따라 전 태스크 일괄 실행 (inc8: 0–2, inc4: 0–6, inc2: 0–15)
# 빠르게 확인만 할 때: TASK_IDS = [0, 1] 처럼 부분 리스트 지정
TASK_IDS = None

# INCREMENT별 마지막 task 인덱스 (학습 시 저장된 체크포인트와 일치해야 함)
_LAST_TASK_BY_INC = {8: 2, 4: 6, 2: 15}


def _resolve_task_ids(inc, task_ids):
    if task_ids is not None:
        return list(task_ids)
    last = _LAST_TASK_BY_INC.get(inc)
    if last is None:
        raise ValueError(
            f'INCREMENT={inc}: TASK_IDS를 직접 지정하거나 _LAST_TASK_BY_INC에 항목을 추가하세요.'
        )
    return list(range(last + 1))


TASK_IDS = _resolve_task_ids(INCREMENT, TASK_IDS)

# 로그 경로: RUN_NAME의 inc*, herding/seed_* 가 SEED·INCREMENT에 맞춰짐
RUN_NAME = (
    f'mmea_tbn_mand_mand_fusion_rgbgyroacce_ep50_bs8_pb1_fr0_inc{INCREMENT}_mem320_train'
)
WEIGHTS_DIR = os.path.join(
    'logs', RUN_NAME, 'herding', f'seed_{SEED}', 'weights', 'beta0.005_lambda0.4'
)

MODALITIES = ['RGB', 'Gyro', 'Acce']
OOD_METHODS = [
    ('Main Only',                    'MaxLogit_Baseline'),
    ('Uniform Sum',                  'MaxLogit_Hybrid_UniformSum'),
    ('Uniform Average',  'MaxLogit_Hybrid_UniformAverage'),
]

# Times New Roman 폰트
_fnames = {f.name for f in fm.fontManager.ttflist}
FONT = 'Times New Roman' if any('Times' in n and 'Roman' in n for n in _fnames) else 'Liberation Serif'

_missing_ckpt = []
for _tid in TASK_IDS:
    _p = os.path.join(WEIGHTS_DIR, f'task_{_tid}_checkpoint_{_tid}.pkl')
    if not os.path.isfile(_p):
        _missing_ckpt.append(_p)
if _missing_ckpt:
    raise FileNotFoundError('체크포인트 없음:\n' + '\n'.join(_missing_ckpt))

print(f'✅ TASK_IDS ({len(TASK_IDS)}): {TASK_IDS}')
print(f'   weights dir: {WEIGHTS_DIR}')
print(f'   Seed={SEED}, Inc={INCREMENT}, Device={"cuda" if torch.cuda.is_available() else "cpu"}')

✅ TASK_IDS (16): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   weights dir: logs/mmea_tbn_mand_mand_fusion_rgbgyroacce_ep50_bs8_pb1_fr0_inc2_mem320_train/herding/seed_1997/weights/beta0.005_lambda0.4
   Seed=1997, Inc=2, Device=cuda


In [10]:
# ===== 셀 2: 모델 로드 =====
import dataloader.data_manager as dm
from utils.utils import set_random_seed, set_device
from models.model_factory import get_model
from dataloader.data_manager import TBNDataManager

# TBNDummyDataset.get 패치: 일부 샘플 로딩 실패 시 빈 텐서 반환
if not hasattr(dm.TBNDummyDataset, '_original_get'):
    dm.TBNDummyDataset._original_get = dm.TBNDummyDataset.get
def _safe_get(self, modality, record, indices):
    try:
        return dm.TBNDummyDataset._original_get(self, modality, record, indices)
    except TypeError:
        if modality in ['RGB', 'RGBDiff', 'Flow']:
            from PIL import Image
            n = len(indices) * getattr(self, 'new_length', {}).get(modality, 1)
            return self.transform[modality]([Image.new('RGB', (224, 224), (0, 0, 0))] * n)
        if modality in ['Gyro', 'Acce']:
            return torch.zeros(self.num_segments, 3, 24, 1)
        raise
dm.TBNDummyDataset.get = _safe_get

# Config 구성 (args.txt에서 실제 학습 설정 로드 후 eval 모드로 오버라이드)
args_path = os.path.join(os.path.dirname(os.path.dirname(WEIGHTS_DIR)), 'args.txt')
base_cfg  = ast.literal_eval(open(args_path).read()) if os.path.isfile(args_path) else json.load(open('exps/exp_mmea_tbn_mand.json'))

cfg = {**base_cfg}
cfg.update(
    seed           = SEED,
    increment      = INCREMENT,
    init_cls       = INCREMENT,
    morst_lambda   = 0.4,
    morst_beta     = 0.005,
    energy_norm_method = 'zscore',
    confidence_method  = 'energy',
    mode       = 'eval',
    enable_ood = True,
    ood_methods= [m[1] for m in OOD_METHODS],
    modality   = MODALITIES,
    workers    = 4,
    use_wandb  = False,
    debug_mode = False,
    device     = [0],
    shuffle    = False,
)

set_random_seed(SEED)
cfg['device'] = set_device(cfg['device'])
model = get_model(cfg['model_name'], cfg)

# 노트북 OOD_METHODS와 평가 시 사용하는 ood_methods 강제 동기화
# (셀 1만 재실행하고 셀 2를 건너뛰면 이전 4종 설정이 남아 Adaptive Reliability가 다시 돌아감)
model.args['ood_methods'] = [m[1] for m in OOD_METHODS]
tmpl  = {m: '{:06d}.jpg' for m in MODALITIES if m in ['RGB', 'RGBDiff']}
data_manager = TBNDataManager(model, tmpl, cfg)
model.total_classnum = data_manager.get_total_classnum()
if isinstance(model._device, list):
    model._device = model._device[0]

# build_rehearsal_memory 패치: memory가 None이면 스킵
_orig_build_mem = model.build_rehearsal_memory
def _patched_build_mem(dm_obj, per_class):
    if model._get_memory() is None:
        return
    _orig_build_mem(dm_obj, per_class)
model.build_rehearsal_memory = _patched_build_mem

print(f'✅ 모델 로드 완료: {cfg["model_name"]} / fusion={cfg["fusion_type"]}')
print(f'   총 클래스 수: {model.total_classnum}')
print(f"   OOD methods (eval): {model.args['ood_methods']}")

Init Gyro model weight
Done. Gyro model ready.
Init Acce model weight
Done. Acce model ready.


🔍 BaselineTBN Debug:
   Modality count: 3
   Backbone feature_dim: 1024
   After fusion feature_dim: 512
🔧 Set fusion.num_segments = 8
✅ Baseline Model Configuration
----------------------------------------
  Backbone:        tbn
  Fusion:          mand_fusion
  Modality:        ['RGB', 'Gyro', 'Acce']
  Segments:        8
  Dropout:         0.5
  Consensus:       avg
video number:5206
video number:1316
✅ 모델 로드 완료: tbn_mand / fusion=mand_fusion
   총 클래스 수: 32
   OOD methods (eval): ['MaxLogit_Baseline', 'MaxLogit_Hybrid_UniformSum', 'MaxLogit_Hybrid_UniformAverage']


In [11]:
# ===== 셀 3: 추론 및 점수 수집 (TASK_IDS 전체) =====
# 모델은 셀 2에서 1회만 로드; 태스크마다 체크포인트만 바꿔 평가

ALL_SCORES = {}

for TASK_ID in TASK_IDS:
    CHECKPOINT = os.path.join(WEIGHTS_DIR, f'task_{TASK_ID}_checkpoint_{TASK_ID}.pkl')
    print(f"\n{'='*16} Task {TASK_ID} {'='*16}")

    cl_results, ood_results, score_distributions = model.inference_mode_evaluation(
        data_manager, CHECKPOINT, TASK_ID
    )

    print('📋 평가 결과')
    print(f'  CL results : {cl_results}')
    print(f'  OOD methods in score_distributions: {list(score_distributions.keys())}')

    scores = {}
    for display_name, method_name in OOD_METHODS:
        if method_name not in score_distributions:
            print(f'  ⚠️  {method_name} not in score_distributions, skip')
            continue
        sd = score_distributions[method_name]
        id_s = np.asarray(sd.get('id_scores', sd.get('id', []))).flatten()
        ood_s = np.asarray(sd.get('ood_scores', sd.get('ood', []))).flatten()
        if len(id_s) == 0 or len(ood_s) == 0:
            print(f'  ⚠️  {display_name}: empty scores')
            continue
        labels = np.concatenate([np.ones(len(id_s)), np.zeros(len(ood_s))])
        all_s = np.concatenate([id_s, ood_s])
        auroc = roc_auc_score(labels, all_s) * 100
        fpr, tpr, _ = roc_curve(labels, all_s)
        fpr95 = fpr[min(np.searchsorted(tpr, 0.95), len(fpr) - 1)] * 100
        scores[display_name] = {
            'id': id_s, 'ood': ood_s, 'metrics': {'AUROC': auroc, 'FPR95': fpr95}
        }
        print(
            f'  ✅ {display_name:20s} | ID n={len(id_s):4d}, OOD n={len(ood_s):4d} '
            f'| AUROC={auroc:.2f}%  FPR95={fpr95:.2f}%'
        )

    ALL_SCORES[TASK_ID] = scores

print(f"\n✅ 셀 3 완료: {len(ALL_SCORES)}개 태스크 점수 수집")


================ Task 0 ================
🔍 TBNClassification Debug:
   Modality count: 3
   Feature dim: 512
   FC input dim: 512
   FC output dim: 2


  📊 Processing ID data (logits + features)...


  🎯 Processing OOD data (logits + features)...


✅ Data extracted - ID: logitstorch.Size([83, 2]), features(664, 512)
                   OOD: logitstorch.Size([1233, 2]), features(9864, 512)
  🔥 UnifiedOODDetector Hybrid methods detected - collecting auxiliary outputs...


  ✅ Auxiliary outputs collected:
     - Main logits: ✅
     - Auxiliary logits: ['RGB', 'Gyro', 'Acce']
     - Confidences: ['RGB', 'Gyro', 'Acce']


...

✅ Data extracted - ID: logitstorch.Size([1235, 30]), features(9880, 512)
                   OOD: logitstorch.Size([81, 30]), features(648, 512)
  🔥 UnifiedOODDetector Hybrid methods detected - collecting auxiliary outputs...


  ✅ Auxiliary outputs collected:
     - Main logits: ✅
     - Auxiliary logits: ['RGB', 'Gyro', 'Acce']
     - Confidences: ['RGB', 'Gyro', 'Acce']


OOD Methods: 100%|██████████| 3/3 [00:00<00:00, 94.28it/s]


📋 평가 결과
  CL results : {'cnn': {'grouped': {'00-01': 74.7, '02-03': 71.95, '04-05': 75.61, '06-07': 93.67, '08-09': 81.61, '10-11': 91.11, '12-13': 92.41, '14-15': 80.49, '16-17': 91.14, '18-19': 76.74, '20-21': 92.41, '22-23': 74.07, '24-25': 86.75, '26-27': 87.65, '28-29': 96.34}, 'top1': 84.37}, 'nme': {'grouped': {'00-01': 73.49, '02-03': 71.95, '04-05': 79.27, '06-07': 92.41, '08-09': 78.16, '10-11': 94.44, '12-13': 88.61, '14-15': 85.37, '16-17': 89.87, '18-19': 75.58, '20-21': 88.61, '22-23': 69.14, '24-25': 85.54, '26-27': 91.36, '28-29': 95.12}, 'top1': 83.89}}
  OOD methods in score_distributions: ['MaxLogit_Baseline', 'MaxLogit_Hybrid_UniformSum', 'MaxLogit_Hybrid_UniformAverage']
  ✅ Main Only            | ID n=1235, OOD n=  81 | AUROC=55.80%  FPR95=92.59%
  ✅ Uniform Sum          | ID n=1235, OOD n=  81 | AUROC=52.05%  FPR95=85.19%
  ✅ Uniform Average      | ID n=1235, OOD n=  81 | AUROC=54.73%  FPR95=88.89%

================ Task 15 ================
🔍 TBNClassification De

📋 평가 결과
  CL results : {'cnn': {'grouped': {'00-01': 79.52, '02-03': 73.17, '04-05': 69.51, '06-07': 94.94, '08-09': 77.01, '10-11': 92.22, '12-13': 87.34, '14-15': 73.17, '16-17': 88.61, '18-19': 79.07, '20-21': 91.14, '22-23': 69.14, '24-25': 81.93, '26-27': 91.36, '28-29': 65.85, '30-31': 97.53}, 'top1': 81.91}, 'nme': {'grouped': {'00-01': 73.49, '02-03': 68.29, '04-05': 76.83, '06-07': 94.94, '08-09': 77.01, '10-11': 95.56, '12-13': 88.61, '14-15': 82.93, '16-17': 86.08, '18-19': 75.58, '20-21': 89.87, '22-23': 70.37, '24-25': 79.52, '26-27': 85.19, '28-29': 71.95, '30-31': 97.53}, 'top1': 82.07}}
  OOD methods in score_distributions: []
  ⚠️  MaxLogit_Baseline not in score_distributions, skip
  ⚠️  MaxLogit_Hybrid_UniformSum not in score_distributions, skip
  ⚠️  MaxLogit_Hybrid_UniformAverage not in score_distributions, skip

✅ 셀 3 완료: 16개 태스크 점수 수집


In [12]:
# ===== 셀 4: ID/OOD 분포 시각화 (1×3, 태스크마다 파일 저장) =====
# 공통 x축 [-20, 30], density 히스토그램, Known=blue / Novel=red, AUC·FPR95 서브캡션

plt.rcParams.update({
    'font.family': FONT,
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'legend.fontsize': 7,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'lines.linewidth': 1.5,
    'lines.markersize': 4,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'figure.dpi': 300,
})

out_dir = os.path.join(PROJECT_ROOT, 'notebook', 'figures', 'mand_ood_dist', f'inc{INCREMENT}')
os.makedirs(out_dir, exist_ok=True)

X_MIN, X_MAX = -20, 30
bins_shared = np.linspace(X_MIN, X_MAX, 60)
n_col = len(OOD_METHODS)
panel_w = 9.2 / 4
_show_fig = len(TASK_IDS) == 1

for TASK_ID, scores in ALL_SCORES.items():
    base = f'mand_inc{INCREMENT}_task{TASK_ID}_seed{SEED}_1x3'

    fig, axes = plt.subplots(1, n_col, figsize=(panel_w * n_col, 2.35))
    if n_col == 1:
        axes = np.array([axes])

    plot_entries = []
    global_ymax = 0.0

    for i, (display_name, method_key) in enumerate(OOD_METHODS):
        ax = axes[i]
        if display_name not in scores:
            ax.set_axis_off()
            plot_entries.append(None)
            continue
        id_raw = np.asarray(scores[display_name]['id']).ravel()
        ood_raw = np.asarray(scores[display_name]['ood']).ravel()
        id_s = id_raw[(id_raw >= X_MIN) & (id_raw <= X_MAX)]
        ood_s = ood_raw[(ood_raw >= X_MIN) & (ood_raw <= X_MAX)]

        ax.hist(id_s, bins=bins_shared, alpha=0.6, label='Known', color='blue', density=True)
        ax.hist(ood_s, bins=bins_shared, alpha=0.6, label='Novel', color='red', density=True)

        if len(id_s) > 0:
            h_id, _ = np.histogram(id_s, bins=bins_shared, density=True)
            global_ymax = max(global_ymax, float(h_id.max()))
        if len(ood_s) > 0:
            h_ood, _ = np.histogram(ood_s, bins=bins_shared, density=True)
            global_ymax = max(global_ymax, float(h_ood.max()))

        plot_entries.append(display_name)

    for i, name in enumerate(plot_entries):
        if name is None:
            continue
        ax = axes[i]
        m = scores[name]['metrics']
        y_hi = global_ymax * 1.08 if global_ymax > 0 else None
        if y_hi is not None:
            ax.set_ylim(0, y_hi)

        ax.set_xlim(X_MIN, X_MAX)
        ax.legend(loc='upper right', handlelength=0.8, handletextpad=0.4,
                  borderpad=0.3, labelspacing=0.25, prop={'family': FONT, 'size': 7})
        ax.set_xlabel('Novelty Score', fontsize=11, fontfamily=FONT)
        if i == 0:
            ax.set_ylabel('Density', labelpad=6, fontsize=11, fontfamily=FONT)

        ax.text(
            0.5, 1.12, name,
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=11, fontweight='bold', fontfamily=FONT, clip_on=False,
        )
        ax.text(
            0.5, 1.02, f"AUC: {m['AUROC']:.1f}%, FPR95: {m['FPR95']:.1f}%",
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=9, fontfamily=FONT, clip_on=False,
        )
        ax.grid(axis='both', linestyle='--', alpha=0.35)

    fig.subplots_adjust(left=0.08, right=0.995, top=0.82, bottom=0.23, wspace=0.30)

    out_pdf = os.path.join(out_dir, f'{base}.pdf')
    out_png = os.path.join(out_dir, f'{base}.png')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.savefig(out_png, bbox_inches='tight', dpi=300)
    print(f'저장됨: {out_pdf}\n        {out_png}')

    if _show_fig:
        plt.show()
    else:
        plt.close(fig)

if not _show_fig:
    print(f'✅ 총 {len(ALL_SCORES)}개 태스크 figure 저장 완료 (배치 모드: plt.show 생략)')


저장됨: /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task0_seed1997_1x3.pdf
        /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task0_seed1997_1x3.png
저장됨: /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task1_seed1997_1x3.pdf
        /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task1_seed1997_1x3.png
저장됨: /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task2_seed1997_1x3.pdf
        /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task2_seed1997_1x3.png
저장됨: /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task3_seed1997_1x3.pdf
        /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task3_seed1997_1x3.png
저장됨: /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task4_seed1997_1x3.pdf
        /workspace/MMEA-MoAS/notebook/figures/mand_ood_dist/inc2/mand_inc2_task4_seed1997_1x3.png
저장됨: /workspace/MMEA-MoAS/notebook/